# Feature Engineering

This notebook creates the final feature set for modeling based on insights from EDA.

## Features to Create
1. **Temporal features**: hour, dayofweek, month, is_weekend, cyclical encodings
2. **VTAT features**: imputed values, buckets, high_vtat flag
3. **Location features**: group to top 10 + Other, then target encoding
4. **Vehicle type**: label encoding

## Setup

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

In [2]:
# Load all splits
train_df = pd.read_parquet("../data/bronze/train.parquet", engine='pyarrow')
val_df = pd.read_parquet("../data/bronze/val.parquet", engine='pyarrow')
test_df = pd.read_parquet("../data/bronze/test.parquet", engine='pyarrow')

print(f"Train shape: {train_df.shape}")
print(f"Val shape: {val_df.shape}")
print(f"Test shape: {test_df.shape}")

Train shape: (112705, 13)
Val shape: (25045, 13)
Test shape: (12250, 13)


## Feature Engineering Functions

In [3]:
def create_temporal_features(input_df):
    result_df = input_df.copy()
    
    # Create datetime
    result_df['datetime'] = pd.to_datetime(result_df['date'] + " " + result_df['time'], format="%Y-%m-%d %H:%M:%S")
    
    # Extract components
    result_df['hour'] = result_df['datetime'].dt.hour
    result_df['dayofweek'] = result_df['datetime'].dt.dayofweek
    result_df['month'] = result_df['datetime'].dt.month
    result_df['day'] = result_df['datetime'].dt.day
    
    # Binary features
    result_df['is_weekend'] = (result_df['dayofweek'] >= 5).astype(int)
    result_df['is_morning_rush'] = ((result_df['hour'] >= 7) & (result_df['hour'] <= 10)).astype(int)
    result_df['is_evening_rush'] = ((result_df['hour'] >= 17) & (result_df['hour'] <= 21)).astype(int)
    result_df['is_peak_hour'] = (result_df['is_morning_rush'] | result_df['is_evening_rush']).astype(int)
    result_df['is_late_night'] = ((result_df['hour'] >= 23) | (result_df['hour'] <= 5)).astype(int)
    
    # Cyclical encoding for hour (24-hour cycle)
    result_df['hour_sin'] = np.sin(2 * np.pi * result_df['hour'] / 24)
    result_df['hour_cos'] = np.cos(2 * np.pi * result_df['hour'] / 24)
    
    # Cyclical encoding for day of week (7-day cycle)
    result_df['dow_sin'] = np.sin(2 * np.pi * result_df['dayofweek'] / 7)
    result_df['dow_cos'] = np.cos(2 * np.pi * result_df['dayofweek'] / 7)
    
    # Cyclical encoding for month (12-month cycle)
    result_df['month_sin'] = np.sin(2 * np.pi * result_df['month'] / 12)
    result_df['month_cos'] = np.cos(2 * np.pi * result_df['month'] / 12)
    
    return result_df

In [4]:
def create_vtat_features(input_df, train_medians=None):
    result_df = input_df.copy()
    
    # Calculate medians from training data if not provided
    if train_medians is None:
        train_medians = result_df.groupby('vehicle_type', observed=True)['avg_vtat'].median().to_dict()
        return_medians = True
    else:
        return_medians = False
    
    # Impute missing VTAT with median by vehicle type
    result_df['avg_vtat_imputed'] = result_df.apply(
        lambda row: row['avg_vtat'] if pd.notna(row['avg_vtat']) 
        else train_medians.get(row['vehicle_type'], result_df['avg_vtat'].median()),
        axis=1
    )
    
    # Create VTAT bucket
    bins = [0, 5, 10, 15, 20]
    labels = [0, 1, 2, 3]  # Numeric labels for modeling
    result_df['vtat_bucket'] = pd.cut(result_df['avg_vtat_imputed'], bins=bins, labels=labels).astype(float)
    
    # High VTAT flag (critical threshold from EDA)
    result_df['is_high_vtat'] = (result_df['avg_vtat_imputed'] >= 15).astype(int)
    
    if return_medians:
        return result_df, train_medians
    return result_df

In [5]:
def group_infrequent_locations(input_df, column, top_n=10, top_locations=None):

    result_df = input_df.copy()
    new_col_name = f"{column}_grouped"
    
    if top_locations is None:
        # Get top N locations from training data
        top_locations = result_df[column].value_counts().head(top_n).index.tolist()
        return_top = True
    else:
        return_top = False
    
    # Map locations: keep top N, replace others with 'Other'
    result_df[new_col_name] = result_df[column].apply(
        lambda x: x if x in top_locations else 'Other'
    )
    
    if return_top:
        return result_df, top_locations
    return result_df

In [6]:
def create_target_encoding(input_df, column, target_col='is_cancelled', train_means=None, smoothing=10):

    if train_means is None:
        # Calculate from training data with smoothing
        global_mean = input_df[target_col].mean()
        agg = input_df.groupby(column, observed=True)[target_col].agg(['mean', 'count'])
        
        # Smoothed mean: (count * mean + smoothing * global_mean) / (count + smoothing)
        smoothed_means = (agg['count'] * agg['mean'] + smoothing * global_mean) / (agg['count'] + smoothing)
        means_dict = smoothed_means.to_dict()
        means_dict['__global__'] = global_mean
        
        encoded = input_df[column].map(means_dict).fillna(global_mean)
        return encoded, means_dict
    else:
        global_mean = train_means.get('__global__', 0.32)
        encoded = input_df[column].map(train_means).fillna(global_mean)
        return encoded

In [7]:
def create_vehicle_encoding(input_df, train_encoder=None):
    result_df = input_df.copy()
    
    if train_encoder is None:
        encoder = LabelEncoder()
        result_df['vehicle_type_encoded'] = encoder.fit_transform(result_df['vehicle_type'].astype(str))
        return result_df, encoder
    else:
        # Handle unseen categories
        result_df['vehicle_type_encoded'] = result_df['vehicle_type'].astype(str).apply(
            lambda x: train_encoder.transform([x])[0] if x in train_encoder.classes_ else -1
        )
        return result_df

## Apply Feature Engineering

In [8]:
# Step 1: Remove leaking columns
leaking_cols = ["avg_ctat", "booking_value", "ride_distance", "payment_method"]

for col in leaking_cols:
    if col in train_df.columns:
        train_df = train_df.drop(columns=[col])
    if col in val_df.columns:
        val_df = val_df.drop(columns=[col])
    if col in test_df.columns:
        test_df = test_df.drop(columns=[col])

print("Leaking columns removed.")

Leaking columns removed.


In [9]:
# Step 2: Create temporal features
train_df = create_temporal_features(train_df)
val_df = create_temporal_features(val_df)
test_df = create_temporal_features(test_df)

print("Temporal features created.")

Temporal features created.


In [10]:
# Step 3: Create VTAT features (fit on train, transform val/test)
train_df, vtat_medians = create_vtat_features(train_df)
val_df = create_vtat_features(val_df, train_medians=vtat_medians)
test_df = create_vtat_features(test_df, train_medians=vtat_medians)

print(f"VTAT features created. Medians by vehicle type: {vtat_medians}")

VTAT features created. Medians by vehicle type: {'Auto': 8.199999809265137, 'Bike': 8.300000190734863, 'Go Mini': 8.199999809265137, 'Go Sedan': 8.199999809265137, 'Premier Sedan': 8.199999809265137, 'Uber XL': 8.399999618530273, 'eBike': 8.300000190734863}


In [11]:
# Step 4: Group infrequent locations into 'Other' (top 10 only)
print("\nGrouping locations (keeping top 10, rest -> 'Other')...")

# Pickup locations
train_df, top_pickup_locations = group_infrequent_locations(train_df, 'pickup_location', top_n=10)
val_df = group_infrequent_locations(val_df, 'pickup_location', top_locations=top_pickup_locations)
test_df = group_infrequent_locations(test_df, 'pickup_location', top_locations=top_pickup_locations)

print(f"\nTop 10 pickup locations: {top_pickup_locations}")

# Drop locations
train_df, top_drop_locations = group_infrequent_locations(train_df, 'drop_location', top_n=10)
val_df = group_infrequent_locations(val_df, 'drop_location', top_locations=top_drop_locations)
test_df = group_infrequent_locations(test_df, 'drop_location', top_locations=top_drop_locations)

print(f"\nTop 10 drop locations: {top_drop_locations}")

# Show distribution after grouping
print(f"\nPickup location distribution after grouping:")
print(train_df['pickup_location_grouped'].value_counts())

print(f"\nDrop location distribution after grouping:")
print(train_df['drop_location_grouped'].value_counts())


Grouping locations (keeping top 10, rest -> 'Other')...

Top 10 pickup locations: ['Barakhamba Road', 'Pragati Maidan', 'Badarpur', 'Dwarka Sector 21', 'AIIMS', 'Pataudi Chowk', 'Mehrauli', 'Jasola', 'Tilak Nagar', 'Madipur']

Top 10 drop locations: ['Narsinghpur', 'Kalkaji', 'Lok Kalyan Marg', 'Ashram', 'Nehru Place', 'Punjabi Bagh', 'Preet Vihar', 'Kashmere Gate ISBT', 'Udyog Vihar', 'Sushant Lok']

Pickup location distribution after grouping:
pickup_location_grouped
Other               105731
Barakhamba Road        724
Pragati Maidan         714
Badarpur               708
Dwarka Sector 21       697
AIIMS                  694
Pataudi Chowk          693
Jasola                 687
Mehrauli               687
Tilak Nagar            685
Madipur                685
Name: count, dtype: int64

Drop location distribution after grouping:
drop_location_grouped
Other                 105749
Narsinghpur              709
Kalkaji                  707
Lok Kalyan Marg          706
Ashram              

In [12]:
# Step 5: Create target encoding for grouped locations (fit on train)
train_df['pickup_encoded'], pickup_means = create_target_encoding(train_df, 'pickup_location_grouped')
train_df['drop_encoded'], drop_means = create_target_encoding(train_df, 'drop_location_grouped')

val_df['pickup_encoded'] = create_target_encoding(val_df, 'pickup_location_grouped', train_means=pickup_means)
val_df['drop_encoded'] = create_target_encoding(val_df, 'drop_location_grouped', train_means=drop_means)

test_df['pickup_encoded'] = create_target_encoding(test_df, 'pickup_location_grouped', train_means=pickup_means)
test_df['drop_encoded'] = create_target_encoding(test_df, 'drop_location_grouped', train_means=drop_means)

print("Location target encoding created.")
print(f"\nPickup encoding means: {pickup_means}")
print(f"\nDrop encoding means: {drop_means}")

Location target encoding created.

Pickup encoding means: {'AIIMS': 0.31705761463804677, 'Badarpur': 0.31505368911456266, 'Barakhamba Road': 0.3136356187126617, 'Dwarka Sector 21': 0.32702766546434187, 'Jasola': 0.31593766466445866, 'Madipur': 0.29670294943473324, 'Mehrauli': 0.33171959753700103, 'Other': 0.32081413658662233, 'Pataudi Chowk': 0.3360007913573877, 'Pragati Maidan': 0.3455919335230938, 'Tilak Nagar': 0.31540800187227536, '__global__': np.float32(0.32085532)}

Drop encoding means: {'Ashram': 0.3008547116363986, 'Kalkaji': 0.3350189008556483, 'Kashmere Gate ISBT': 0.30302517413237856, 'Lok Kalyan Marg': 0.32850357823531723, 'Narsinghpur': 0.318787974301565, 'Nehru Place': 0.3440462454984134, 'Other': 0.3207500974509155, 'Preet Vihar': 0.31503370826336446, 'Punjabi Bagh': 0.32790922346885965, 'Sushant Lok': 0.31926515124652577, 'Udyog Vihar': 0.33171959753700103, '__global__': np.float32(0.32085532)}


In [13]:
# Step 6: Create vehicle type encoding
train_df, vehicle_encoder = create_vehicle_encoding(train_df)
val_df = create_vehicle_encoding(val_df, train_encoder=vehicle_encoder)
test_df = create_vehicle_encoding(test_df, train_encoder=vehicle_encoder)

print(f"Vehicle encoding created. Classes: {vehicle_encoder.classes_}")

Vehicle encoding created. Classes: ['Auto' 'Bike' 'Go Mini' 'Go Sedan' 'Premier Sedan' 'Uber XL' 'eBike']


## Select Final Features

In [14]:
# Define feature columns for modeling
FEATURE_COLS = [
    # VTAT features (most important)
    'avg_vtat_imputed',
    'vtat_bucket',
    'is_high_vtat',
    
    # Location features (target encoded, grouped to top 10 + Other)
    'pickup_encoded',
    'drop_encoded',
    
    # Vehicle type
    'vehicle_type_encoded',
    
    # Temporal features
    'hour',
    'dayofweek',
    'month',
    'is_weekend',
    'is_peak_hour',
    'is_late_night',
    
    # Cyclical encodings
    'hour_sin',
    'hour_cos',
    'dow_sin',
    'dow_cos',
    'month_sin',
    'month_cos',
]

TARGET_COL = 'is_cancelled'

print(f"Total features: {len(FEATURE_COLS)}")
print(f"Features: {FEATURE_COLS}")

Total features: 18
Features: ['avg_vtat_imputed', 'vtat_bucket', 'is_high_vtat', 'pickup_encoded', 'drop_encoded', 'vehicle_type_encoded', 'hour', 'dayofweek', 'month', 'is_weekend', 'is_peak_hour', 'is_late_night', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos']


In [15]:
# Create final datasets
X_train = train_df[FEATURE_COLS].copy()
y_train = train_df[TARGET_COL].copy()

X_val = val_df[FEATURE_COLS].copy()
y_val = val_df[TARGET_COL].copy()

X_test = test_df[FEATURE_COLS].copy()
y_test = test_df[TARGET_COL].copy()

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"X_test shape: {X_test.shape}")

X_train shape: (112705, 18)
X_val shape: (25045, 18)
X_test shape: (12250, 18)


In [16]:
# Check for any remaining missing values
print("\nMissing values in X_train:")
print(X_train.isnull().sum())


Missing values in X_train:
avg_vtat_imputed        0
vtat_bucket             0
is_high_vtat            0
pickup_encoded          0
drop_encoded            0
vehicle_type_encoded    0
hour                    0
dayofweek               0
month                   0
is_weekend              0
is_peak_hour            0
is_late_night           0
hour_sin                0
hour_cos                0
dow_sin                 0
dow_cos                 0
month_sin               0
month_cos               0
dtype: int64


In [21]:
# Final data summary
print("FINAL DATASET SUMMARY")
print("-" * 60)
print(f"\nTraining set:")
print(f"  Samples: {len(X_train):,}")
print(f"  Features: {X_train.shape[1]}")
print(f"  Cancellation rate: {y_train.mean():.2%}")

print(f"\nValidation set:")
print(f"  Samples: {len(X_val):,}")
print(f"  Cancellation rate: {y_val.mean():.2%}")

print(f"\nTest set:")
print(f"  Samples: {len(X_test):,}")
print(f"  Cancellation rate: {y_test.mean():.2%}")

FINAL DATASET SUMMARY
------------------------------------------------------------

Training set:
  Samples: 112,705
  Features: 18
  Cancellation rate: 32.09%

Validation set:
  Samples: 25,045
  Cancellation rate: 31.66%

Test set:
  Samples: 12,250
  Cancellation rate: 31.90%


## Save Processed Data

In [ ]:
import os

output_dir = "../data/silver"
os.makedirs(output_dir, exist_ok=True)

X_train.to_parquet(os.path.join(output_dir, "X_train.parquet"), engine='pyarrow', index=False)
X_val.to_parquet(os.path.join(output_dir, "X_val.parquet"), engine='pyarrow', index=False)
X_test.to_parquet(os.path.join(output_dir, "X_test.parquet"), engine='pyarrow', index=False)

y_train.to_frame().to_parquet(os.path.join(output_dir, "y_train.parquet"), engine='pyarrow', index=False)
y_val.to_frame().to_parquet(os.path.join(output_dir, "y_val.parquet"), engine='pyarrow', index=False)
y_test.to_frame().to_parquet(os.path.join(output_dir, "y_test.parquet"), engine='pyarrow', index=False)

print(f"\nProcessed data saved to {output_dir}/")
print("Files created:")
for f in os.listdir(output_dir):
    print(f"  - {f}")


Processed data saved to ../data/silver/
Files created:
  - y_val.parquet
  - X_val.parquet
  - X_test.parquet
  - encoders.pkl
  - X_train.parquet
  - y_test.parquet
  - y_train.parquet


In [ ]:
import pickle

encoders = {
    'vtat_medians': vtat_medians,
    'top_pickup_locations': top_pickup_locations,
    'top_drop_locations': top_drop_locations,
    'pickup_means': pickup_means,
    'drop_means': drop_means,
    'vehicle_encoder': vehicle_encoder,
    'feature_cols': FEATURE_COLS,
}

with open(os.path.join(output_dir, "encoders.pkl"), 'wb') as f:
    pickle.dump(encoders, f)

print("Encoders saved to encoders.pkl")

Encoders saved to encoders.pkl


## Feature Statistics

In [ ]:
X_train.describe().round(3)

,avg_vtat_imputed,vtat_bucket,is_high_vtat,pickup_encoded,drop_encoded,vehicle_type_encoded,hour,dayofweek,month,is_weekend,is_peak_hour,is_late_night,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos
count,112705.000,112705.000,112705.000,112705.000,112705.000,112705.000,112705.000,112705.000,112705.000,112705.000,112705.000,112705.000,112705.000,112705.000,112705.000,112705.000,112705.000,112705.000
mean,8.431,1.134,0.026,0.321,0.321,2.141,14.038,2.992,5.002,0.285,0.549,0.082,-0.242,-0.212,-0.000,0.003,0.149,-0.266
std,3.638,0.763,0.158,0.003,0.003,1.785,5.422,2.003,2.580,0.452,0.498,0.275,0.715,0.621,0.707,0.708,0.728,0.614
min,2.000,0.000,0.000,0.297,0.301,0.000,0.000,0.000,1.000,0.000,0.000,0.000,-1.000,-1.000,-0.975,-0.901,-1.000,-1.000
25%,5.600,1.000,0.000,0.321,0.321,1.000,10.000,1.000,3.000,0.000,0.000,0.000,-0.866,-0.707,-0.782,-0.901,-0.500,-0.866
50%,8.200,1.000,0.000,0.321,0.321,2.000,15.000,3.000,5.000,0.000,1.000,0.000,-0.500,-0.259,0.000,-0.223,0.500,-0.500
75%,11.000,2.000,0.000,0.321,0.321,3.000,18.000,5.000,7.000,1.000,1.000,0.000,0.500,0.259,0.782,0.623,0.866,0.000
max,20.000,3.000,1.000,0.346,0.344,6.000,23.000,6.000,9.000,1.000,1.000,1.000,1.000,1.000,0.975,1.000,1.000,0.866
